# 01 - Ingesta

Pasamos los CSV de `/nyc/raw` a Parquet en `/nyc/bronze` sin cambiarles nada. Se corre una sola vez, después de `scripts/descargar_datos.sh`.

| Capa | Ruta | Contenido |
|---|---|---|
| cruda | `/nyc/raw/<fuente>/` | CSV originales |
| bronce | `/nyc/bronze/<fuente>/` | igual pero en Parquet, todo texto (este notebook) |
| plata | `/nyc/silver/<fuente>/` | tipos, filtros y limpieza (04 y 05) |
| oro | `/nyc/gold/<tabla>/` | agregados para el análisis y los modelos |

Usamos Parquet porque guarda los datos por columnas, viene comprimido y Spark lo puede leer en paralelo.

## 1. Sesión y rutas

In [ ]:
import re
import time
import pandas as pd
from pyspark.sql import functions as F

from config import RAW, BRONZE, sesion

spark = sesion("01_ingesta")
sc = spark.sparkContext

# HDFS desde python
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
Path = jvm.org.apache.hadoop.fs.Path

def tamano_hdfs(ruta):
    return fs.getContentSummary(Path(ruta)).getLength()

print("Master:", sc.master, "| paralelismo:", sc.defaultParallelism)
print("HDFS  :", fs.getUri())

## 2. Manifiesto

El manifiesto lo deja el script de descarga: fecha, tamaño y número de líneas de cada archivo.

In [ ]:
manifiesto = (
    spark.read.option("header", True).option("sep", "\t")
    .csv(f"{RAW}/manifiesto.tsv")
    .toPandas()
    .astype({"bytes": "int64", "lineas": "int64"})
)
manifiesto["tamano_MB"] = (manifiesto["bytes"] / 1024**2).round(1)
manifiesto[["fuente", "id", "descargado_utc", "tamano_MB", "lineas"]]

## 3. Lectura

- Todo se lee como texto, sin `inferSchema`. Si no, se pierden cosas como los ceros a la izquierda de los códigos postales, y además los puntajes del SAT traen `"s"`. Los tipos se ponen después, en plata.
- Hay campos con saltos de línea, por eso va `multiLine=True`. El problema es que así cada CSV lo lee una sola tarea, entonces antes de escribir se hace un `repartition`.
- Los nombres de las columnas quedan en minúscula y con guion bajo.

In [ ]:
def normalizar(nombre):
    n = nombre.strip().lower()
    n = re.sub(r"[^0-9a-z]+", "_", n).strip("_")
    return n or "columna"

def normalizar_columnas(df):
    vistos, nuevos = {}, []
    for c in df.columns:
        n = normalizar(c)
        if n in vistos:          # por si se repiten
            vistos[n] += 1
            n = f"{n}_{vistos[n]}"
        else:
            vistos[n] = 0
        nuevos.append(n)
    return df.toDF(*nuevos)

[normalizar(c) for c in ["CRASH DATE", "NUMBER OF PERSONS INJURED", "Lon_Lat", "SAT Math Avg. Score"]]

## 4. CSV a Parquet

Más o menos una partición por cada 128 MB, y nunca menos particiones que cores.

In [ ]:
FUENTES_CSV = [f for f in manifiesto["fuente"] if f != "puma_2010"]   # puma es geojson

def ingerir(fuente):
    t0 = time.time()
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .option("inferSchema", False)
        .csv(f"{RAW}/{fuente}/")
    )
    df = normalizar_columnas(df)
    bytes_csv = tamano_hdfs(f"{RAW}/{fuente}")
    particiones = max(sc.defaultParallelism, int(bytes_csv / (128 * 1024**2)) + 1)
    df.repartition(particiones).write.mode("overwrite").parquet(f"{BRONZE}/{fuente}")

    bronce = spark.read.parquet(f"{BRONZE}/{fuente}")
    filas = bronce.count()
    lineas = int(manifiesto.loc[manifiesto["fuente"] == fuente, "lineas"].iloc[0])
    bytes_pq = tamano_hdfs(f"{BRONZE}/{fuente}")
    return {
        "fuente": fuente,
        "filas": filas,
        "columnas": len(bronce.columns),
        "lineas_csv": lineas,
        "registros_multilinea": (lineas - 1) - filas,
        "CSV_MB": round(bytes_csv / 1024**2, 1),
        "Parquet_MB": round(bytes_pq / 1024**2, 1),
        "compresion_x": round(bytes_csv / bytes_pq, 1),
        "particiones": particiones,
        "segundos": round(time.time() - t0, 1),
    }

resumen = []
for fuente in FUENTES_CSV:
    r = ingerir(fuente)
    print(f"{fuente:>20}: {r['filas']:>12,} filas en {r['segundos']} s")
    resumen.append(r)

In [ ]:
resumen_df = pd.DataFrame(resumen)
resumen_df

`registros_multilinea` es la diferencia entre las líneas del archivo y los registros que leyó Spark. Si da más de 0 es porque hay campos con saltos de línea.

## 5. Esquema bronce

In [ ]:
for fuente in FUENTES_CSV:
    df = spark.read.parquet(f"{BRONZE}/{fuente}")
    print(f"\n=== {fuente} ({len(df.columns)} columnas) ===")
    print(", ".join(df.columns))

In [ ]:
# ejemplo
spark.read.parquet(f"{BRONZE}/arrestos_historico").limit(5).toPandas()

In [ ]:
resumen_df.to_csv("../figuras/tabla_ingesta.csv", index=False)

## Conclusiones

- Los CSV originales siguen intactos en `/nyc/raw`.
- Bronce tiene lo mismo pero en Parquet, que pesa menos y se lee más rápido.
- El filtro por periodo todavía no se aplica; eso se hace al pasar a plata.